# 05 · Deployment and monitoring

### Session 3 core notebook | Safiri Telecom churn case

---

Your existing notebooks end at `pickle.dump()`. In industry that is roughly the
one-third mark.

This notebook covers what happens next, in the order it goes wrong:

1. The feature you compute at scoring time is not the feature you trained on.
2. The world changes and nobody notices for two quarters.
3. Somebody asks who is accountable for an automated decision, and there is no answer.

### Cell tags

| Tag | What you do |
|---|---|
| **READ** | Run it, read the output, answer the question. No typing. |
| **FILL** | Change one clearly marked value. |
| **BUILD** | Open-ended. Write your own code. |

## 0. Setup

In [ ]:
import numpy as np
import pandas as pd

def make_panel(seed=42, n_subscribers=9000, n_months=8):
    """Simulate a monthly prepaid subscriber panel for Safiri Telecom."""
    rng = np.random.default_rng(seed)
    n = n_subscribers
    region  = rng.choice(["Nairobi","Coast","Western","Rift Valley","Eastern"], n, p=[.34,.16,.18,.20,.12])
    plan    = rng.choice(["Prepaid Basic","Prepaid Plus","Data Bundle"], n, p=[.55,.28,.17])
    handset = rng.choice(["Feature","Smart-Entry","Smart-Mid","Smart-High"], n, p=[.22,.38,.28,.12])
    tenure0, base_arpu = rng.integers(1,72,n), rng.lognormal(5.6,.55,n)
    heavy_data, eps = rng.random(n) < .30, rng.normal(0,.55,n)

    r1 = (-1.0 - 0.032*np.clip(tenure0,0,48) - 0.0014*base_arpu
          + 0.60*(plan=="Prepaid Basic") + 0.50*(handset=="Feature")
          + 0.30*(region=="Western") + eps)
    r2 = (-0.55 - 0.006*np.clip(tenure0,0,48) + 0.0003*base_arpu + 1.15*heavy_data
          + 0.70*np.isin(handset,["Smart-Mid","Smart-High"]) + 0.20*(plan=="Prepaid Basic") + eps)
    p1, p2 = 1/(1+np.exp(-r1)), 1/(1+np.exp(-r2))

    churn_month = np.full(n, 99)
    for m in range(1, n_months+1):
        hz = np.clip((p1 if m <= 5 else p2)*0.20, 0, .9)
        churn_month[(rng.random(n) < hz) & (churn_month == 99)] = m

    rows = []
    for m in range(1, n_months+1):
        alive = churn_month >= m
        decay = np.where(np.clip(churn_month-m,0,6) <= 2, 0.66 + 0.17*np.clip(churn_month-m,0,6), 1.0)
        rows.append(pd.DataFrame({
            "subscriber_id": np.arange(n), "month": m, "region": region, "plan": plan,
            "handset_class": handset, "tenure_months": tenure0 + m,
            "topup_kes": np.where(alive, base_arpu*decay*rng.lognormal(0,.30,n), 0).round(0),
            "topup_count": np.where(alive, rng.poisson(np.clip(6*decay,0.3,None)), 0),
            "data_mb": np.where(alive, base_arpu*decay*rng.lognormal(0,.5,n)*(1+1.4*heavy_data), 0).round(1),
            "voice_minutes": np.where(alive, rng.gamma(3, 22*decay), 0).round(1),
            "days_since_last_topup": np.where(alive,
                np.clip(rng.poisson(np.clip(20*(1-decay)+4,1,None)),0,60), 60),
            "complaints_90d": rng.poisson(0.18 + 0.45*((churn_month-m) <= 2), n),
            "active": alive.astype(int), "_churn_month": churn_month}))
    return pd.concat(rows, ignore_index=True)

def build_snapshot(panel, prediction_month, horizon=2):
    """Features as at the prediction point; label from the outcome window."""
    obs = panel[(panel.month == prediction_month) & (panel.active == 1)].copy()
    cm = obs._churn_month.values
    obs["churned"] = ((cm > prediction_month) & (cm <= prediction_month + horizon)).astype(int)
    return obs.drop(columns=["_churn_month", "active", "month"])

FEATURES_NUM = ["tenure_months","topup_kes","topup_count","data_mb",
                "voice_minutes","days_since_last_topup","complaints_90d"]
FEATURES_CAT = ["region","plan","handset_class"]

import matplotlib.pyplot as plt
import seaborn as sns
import os, pickle, json

from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.metrics import roc_auc_score

sns.set_style("whitegrid")
plt.rcParams["figure.figsize"] = (9.5, 4.5)

panel = make_panel()
train = pd.concat([build_snapshot(panel,3), build_snapshot(panel,4)], ignore_index=True)

pre = ColumnTransformer([("num", StandardScaler(), FEATURES_NUM),
                         ("cat", OneHotEncoder(handle_unknown="ignore"), FEATURES_CAT)])
model = Pipeline([("pre", pre), ("clf", LogisticRegression(max_iter=3000))])
model.fit(train[FEATURES_NUM + FEATURES_CAT], train.churned)
print("Model trained on prediction months 3-4.")

## 1. Serving

A model in production is a function that takes one subscriber and returns one score.
Writing it forces you to confront what the caller must supply, which is where the
first surprises appear.

In [ ]:
REQUIRED = FEATURES_NUM + FEATURES_CAT

def score_subscriber(record: dict) -> dict:
    """Score one subscriber. This is the contract the calling system must satisfy."""
    missing = [c for c in REQUIRED if c not in record]
    if missing:
        return {"status": "error", "reason": f"missing fields: {missing}"}
    row = pd.DataFrame([record])[REQUIRED]
    if row.isnull().any().any():
        return {"status": "error", "reason": "null values in required fields"}
    p = float(model.predict_proba(row)[:, 1][0])
    return {"status": "ok", "churn_probability": round(p, 4),
            "contact": p >= 0.25}

example = train.iloc[0][REQUIRED].to_dict()
print(json.dumps(score_subscriber(example), indent=2))

broken = {k: v for k, v in example.items() if k != "data_mb"}
print(json.dumps(score_subscriber(broken), indent=2))

**Fail loudly, never silently.** The second call returns an error rather than a
plausible-looking score computed from a default.

The alternative, filling a missing field with a zero or a median and returning a
number anyway, is how a broken upstream feed produces six weeks of confident,
meaningless scores that nobody questions because the dashboard stayed green.

## 2. Training-serving skew

The feature was computed one way in the training pipeline and a slightly different
way in the production job. Nothing errors. The scores are simply wrong.

This is the most common production failure in applied ML, and it is invisible
without a deliberate check.

In [ ]:
live = build_snapshot(panel, 6)

# The production feature job was written by a different team. Three small differences,
# each individually defensible, none of them flagged anywhere.
skewed = live.copy()
skewed["data_mb"] = skewed["data_mb"] / 1024                      # they used GB
skewed["days_since_last_topup"] = skewed["days_since_last_topup"].clip(upper=7)    # 7-day window
skewed["complaints_90d"] = (skewed["complaints_90d"] > 0).astype(int)  # made it a flag

clean_scores  = model.predict_proba(live[REQUIRED])[:, 1]
skewed_scores = model.predict_proba(skewed[REQUIRED])[:, 1]

print(f"AUC on correctly computed features: {roc_auc_score(live.churned, clean_scores):.3f}")
print(f"AUC on the production features     : {roc_auc_score(live.churned, skewed_scores):.3f}")

top = 500
overlap = len(set(np.argsort(-clean_scores)[:top]) & set(np.argsort(-skewed_scores)[:top]))
print(f"\nOf the top {top} subscribers - the ONLY ones anyone will call -")
print(f"the two versions agree on {overlap} ({overlap/top:.0%}).")
print("\nNothing raised an error. No dashboard turned red.")

### The defence: assert the training distribution at serving time

In [ ]:
def build_feature_contract(reference: pd.DataFrame) -> dict:
    """Record what each feature looked like in training. Ship this with the model."""
    return {c: {"max": float(reference[c].max()), "mean": float(reference[c].mean()),
                "nunique": int(reference[c].nunique())}
            for c in FEATURES_NUM}

CONTRACT = build_feature_contract(train)

def check_contract(batch: pd.DataFrame, contract=CONTRACT) -> pd.DataFrame:
    """Three cheap checks. One alone is not enough - each catches a different bug."""
    out = []
    for c, ref in contract.items():
        mean_ratio = batch[c].mean() / (ref["mean"] + 1e-9)
        max_ratio  = batch[c].max()  / (ref["max"]  + 1e-9)
        alerts = []
        if abs(mean_ratio - 1) > 0.50: alerts.append("MEAN")      # unit / scale errors
        if abs(max_ratio  - 1) > 0.40: alerts.append("RANGE")     # clipping, truncation
        if batch[c].nunique() < ref["nunique"] * 0.50: alerts.append("GRANULARITY")  # binarised
        out.append({"feature": c, "mean ratio": mean_ratio, "max ratio": max_ratio,
                    "live distinct": batch[c].nunique(), "ALERT": " ".join(alerts)})
    return pd.DataFrame(out).sort_values("ALERT", ascending=False)

print("=== correctly computed features ===")
print(check_contract(live).round(2).to_string(index=False))
print("\n=== production features ===")
print(check_contract(skewed).round(2).to_string(index=False))

**Three checks, and each caught a different bug.**

`MEAN` caught the unit error on `data_mb`, where the production job supplied gigabytes
rather than megabytes. `RANGE` caught `days_since_last_topup` arriving from a 7-day
lookback window when the model was trained on a 60-day one. `GRANULARITY` caught `complaints_90d` arriving as a 0/1 flag
instead of a count.

Notice that no single check would have found all three. A mean comparison misses
clipping, because clipping barely moves a mean. A range check misses a binarisation
that stays inside the original range. This is why monitoring is a small battery of
cheap tests rather than one clever statistic.

Twenty lines of code, run on every scoring batch, turning a silent six-week failure
into an alert on day one. Ship it with the model.

## 3. Drift

Skew is your bug. Drift is the world changing, and it needs a different response.

Three kinds, and they are not interchangeable:

| Kind | What moved | Typical response |
|---|---|---|
| **Data drift** | The inputs. Feature distributions shift. | Investigate. Often benign. |
| **Concept drift** | The relationship between inputs and outcome. | Retrain. Possibly re-frame. |
| **Pipeline breakage** | Nothing real. A join broke, a feed stopped. | Fix the pipeline. Do not retrain. |

The last is by far the most common, and retraining on broken data bakes the breakage
into the model. Diagnose before you react.

In [ ]:
def population_stability_index(expected, actual, bins=10):
    """PSI: the standard drift statistic. <0.1 stable, 0.1-0.25 moderate, >0.25 significant."""
    edges = np.quantile(expected, np.linspace(0, 1, bins + 1))
    edges[0], edges[-1] = -np.inf, np.inf
    e = np.histogram(expected, edges)[0] / len(expected) + 1e-6
    a = np.histogram(actual,   edges)[0] / len(actual)   + 1e-6
    return float(np.sum((a - e) * np.log(a / e)))

psi = pd.DataFrame([{
    "feature": c,
    "PSI": population_stability_index(train[c].values, live[c].values),
} for c in FEATURES_NUM]).sort_values("PSI", ascending=False)
psi["verdict"] = pd.cut(psi.PSI, [-np.inf, .1, .25, np.inf],
                        labels=["stable", "moderate", "significant"])
print(psi.round(4).to_string(index=False))

In [ ]:
# Data drift is only half the question. Has the RELATIONSHIP changed?
early = build_snapshot(panel, 3)
print(f"{'Segment':<26}{'Month 3':>10}{'Month 6':>10}{'change':>10}")
print("-" * 56)
for label, fn in [("Tenure over 48 months", lambda d: d.tenure_months > 48),
                  ("Feature phone",         lambda d: d.handset_class == "Feature"),
                  ("Smart-High handset",    lambda d: d.handset_class == "Smart-High"),
                  ("Heavy data users",      lambda d: d.data_mb > d.data_mb.quantile(.75))]:
    a, b = early[fn(early)].churned.mean(), live[fn(live)].churned.mean()
    print(f"{label:<26}{a:>9.1%}{b:>10.1%}{b/a:>9.1f}x")

print(f"\nOverall churn rate, month 3 -> month 6: {early.churned.mean():.1%} -> {live.churned.mean():.1%}")

**READ and answer.** Look at the AUC first, then at the overlap.

The AUC barely moved, which is exactly why this failure survives so long in the wild.
The number everyone watches looks fine. But the two versions disagree about nearly
half the people who will actually be called, and the call list is the only output
that matters.

**READ and answer.** PSI on the input features is mostly unremarkable. The
relationship table is not.

> *Which of the three kinds of drift is this, and does it call for a retrain or
> something more?*

This is concept drift, and it is the dangerous kind precisely because input
monitoring alone does not catch it. If you only watch feature distributions, this
model degrades for two quarters while every dashboard stays green.

**Monitor outcomes, not just inputs.**

## 4. The retraining trigger

In [ ]:
# A trigger is a rule agreed in advance, so the decision is not made under pressure
# by whoever happens to notice.

TRIGGERS = {
    "scheduled":        "Retrain monthly regardless. Cheap insurance.",
    "performance":      f"Precision@500 on realised outcomes falls below {0.238:.1%} "
                        "(break-even) for two consecutive campaigns.",
    "input drift":      "PSI > 0.25 on any feature in the top 5 by importance.",
    "outcome drift":    "Realised churn rate moves more than 50% from the training period.",
    "contract breach":  "Any feature more than 3 sd from its training mean.",
    "manual":           "Any known change to a source system, tariff, or competitor action.",
}
for k, v in TRIGGERS.items():
    print(f"{k:<18} {v}")

realised, trained = live.churned.mean(), train.churned.mean()
print(f"\nOutcome-drift check: {trained:.1%} -> {realised:.1%} "
      f"({abs(realised/trained - 1):.0%} change)")
print("TRIGGER FIRED: retrain." if abs(realised/trained - 1) > 0.5 else "Within tolerance.")

**Note the lag.** The outcome trigger needs realised churn, which for a 2-month
outcome window means you learn about degradation at least two months after it
started, on top of the time to detect it.

That lag is a property of your label definition, chosen back in notebook 00. Every
decision in this module is still in play at the end of it.

## 5. The model card

In [ ]:
MODEL_CARD = {
    "model": "Safiri Telecom prepaid churn, v1",
    "owner": "[named individual, not a team]",
    "date": "[date]",
    "decision_served": "Selects subscribers for the monthly outbound retention campaign",
    "who_acts": "Consumer Retention call centre, 500 calls per month",
    "label": "No longer active within 2 months of the prediction point",
    "label_owner": "[who agreed this definition]",
    "prediction_point": "Month end; features use only data available at that moment",
    "training_period": "Prediction months 3-4",
    "validation": "Temporal holdout on prediction month 6",
    "baseline": "Incumbent rule: rank by complaints in prior 90 days",
    "performance_vs_baseline": "[state it honestly, including when the baseline wins]",
    "cost_of_error": "FP: KES 200 wasted offer. FN: KES 840 expected margin lost.",
    "known_limitations": [
        "Trained before the month-6 competitor price war; drivers have since changed",
        "Under-serves segments with low historical complaint rates",
        "Feature contract must be checked on every scoring batch",
    ],
    "monitoring": list(TRIGGERS.keys()),
    "human_oversight": "Retention agent may override any individual score",
    "data_protection": "Automated profiling of identifiable individuals. Kenya Data "
                       "Protection Act 2019 applies: lawful basis, purpose limitation, "
                       "and the data subject's right not to be subject to a decision "
                       "based solely on automated processing.",
}
print(json.dumps(MODEL_CARD, indent=2))

### On the last entry

This model profiles identifiable individuals and drives a commercial decision about
them. In Kenya that engages the **Data Protection Act 2019**, which gives a data
subject rights in relation to decisions based solely on automated processing.

Two practical consequences for how you build:

**Keep a human in the loop.** A retention agent who can override a score is not just
good service design, it is the simplest way to ensure no decision is made *solely*
by automated processing.

**Be able to explain an individual decision.** "The gradient boosting model scored
you 0.81" is not an explanation. This is one of several good reasons to prefer the
simpler model when it performs comparably, and in notebook 04 it frequently did.

Confirm the current requirements with counsel or the Office of the Data Protection
Commissioner before deploying anything. This notebook is not legal advice.

---

## 6. Your first 90 days

What you will actually be asked to do, in rough order of frequency:

| Week | What lands on your desk |
|---|---|
| 1-2 | Find out where the data lives and who owns each table. Most of this is conversations. |
| 3-4 | Reproduce somebody else's number. It will not reproduce. Find out why. |
| 5-8 | Your first model. It will be a logistic regression and it will be fine. |
| 9-12 | Something breaks in production. How you handle it determines your reputation. |

**The four habits that separate people who progress:**

1. **Ask what decision it serves before you ask for the data.**
2. **Always report the baseline**, including when it beats you.
3. **Say when you were wrong, the day you find out.** Not the week after.
4. **Write down the label definition**, and make somebody in the business agree to it.

None of those require a better algorithm. All of them require you to have understood
the lifecycle, which is what this module was for.